# Multi-Objective Portfolio Optimization Using Reinforcement Learning
### Major Project — Phase I & II Experiments
**Institution**: IILM University, Greater Noida  
**Team**: Yug Bhandari, Nimisha Mishra, Yojit Bhatt, Jahanvi Jha, Sunny Ranjan  
**Advisor**: Ms. Vishakha Agarwal  

---
This notebook demonstrates the complete step-by-step research pipeline:
1. **Historical Market Data Ingestion & Cleaning**
2. **Financial Feature Engineering & Correlation Analysis**
3. **Market Regime Detection (Bull, Bear, High Volatility)**
4. **Classical Portfolio Benchmarks (Equal Weight, Buy & Hold, Markowitz MVO)**
5. **Deep Reinforcement Learning (PPO) Evaluation**
6. **Ablation Study (Regime-Aware PPO vs Non-Regime PPO)**


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Environment configured. Project root:', PROJECT_ROOT)


## 1. Clean Historical Market Data
We loaded 10 years (2016–2025) of daily data for 5 high-cap tech equities: AAPL, AMZN, GOOGL, MSFT, NVDA.
Let\'s inspect the cleaned price matrix:

In [ ]:
prices = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'processed', 'clean_prices.csv'), index_col=0, parse_dates=True)
print(f'Total trading days: {len(prices)}')
print(f'Date range: {prices.index.min().date()} to {prices.index.max().date()}')
display(prices.head())
display(prices.describe())


In [ ]:
plt.figure(figsize=(12, 6))
# Normalize to start at 100 for visual comparison
norm_prices = (prices / prices.iloc[0]) * 100
for col in norm_prices.columns:
    plt.plot(norm_prices.index, norm_prices[col], label=col, linewidth=2)

plt.title('Historical Normalized Asset Growth (Base = 100, 2016-2025)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Normalized Price')
plt.legend()
plt.tight_layout()
plt.show()


## 2. Financial Feature Engineering & Correlation Heatmap
Let\'s examine daily returns, rolling volatility, and cross-asset correlations.

In [ ]:
features = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'features', 'portfolio_features.csv'), index_col=0, parse_dates=True)
print(f'Engineered features shape: {features.shape}')
display(features.head(3))

# Cross-Asset Correlation Matrix
returns = prices.pct_change().dropna()
corr_matrix = returns.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', vmin=0.3, vmax=1.0, linewidths=1)
plt.title('Cross-Asset Daily Return Correlation Matrix (2016-2025)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Market Regime Detection (Bull, Bear, High Volatility)
Rule-based classification dynamically detects regime transitions:

In [ ]:
regimes = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'features', 'market_regimes.csv'), index_col=0, parse_dates=True)

# Regime distribution
regime_counts = regimes['regime'].value_counts()
print('Regime Distribution:')
display(regime_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Donut Chart
axes[0].pie(regime_counts, labels=regime_counts.index, autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c', '#f39c12'], startangle=140, pctdistance=0.75, wedgeprops=dict(width=0.4))
axes[0].set_title('Market Regime Distribution (2016-2025)', fontweight='bold')

# Volatility over time colored by regime
axes[1].plot(regimes.index, regimes['avg_volatility'] * 100, color='gray', alpha=0.5, label='Volatility')
axes[1].axhline(y=regimes['avg_volatility'].quantile(0.75)*100, color='red', linestyle='--', label='75th Pct High-Vol Threshold')
axes[1].set_title('Average Market Volatility (%) with Threshold', fontweight='bold')
axes[1].set_ylabel('Annualized Volatility (%)')
axes[1].legend()

plt.tight_layout()
plt.show()


## 4. Out-of-Sample Benchmark Comparison (2024-2025)
Comparison between:
1. **Equal Weight (1/N)**
2. **Buy & Hold**
3. **Markowitz Mean-Variance Optimization**
4. **PPO without Regime (Ablation)**
5. **Regime-Aware PPO**

In [ ]:
results_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'research_results', 'experiment_results.csv'))
display(results_df)


In [ ]:
from IPython.display import Image, display

print('--- Out-of-Sample Cumulative Portfolio Growth ---')
display(Image(filename=os.path.join(PROJECT_ROOT, 'data', 'research_results', 'cumulative_returns.png')))

print('--- Out-of-Sample Drawdown Curves (%) ---')
display(Image(filename=os.path.join(PROJECT_ROOT, 'data', 'research_results', 'drawdowns.png')))
